In [19]:
import os
from argparse import ArgumentParser

import torch
import numpy as np

from neuralut.nn import (
    generate_truth_tables,
    lut_inference,
    module_list_to_verilog_module,
)

from train import configs, model_config, test
from torch.utils.data import DataLoader, TensorDataset

import openml
from models import JetSubstructureNeqModel, JetSubstructureLutModel
from neuralut.synthesis import synthesize_and_get_resource_counts

cuda_device = 2
config = configs["jsc-openml"]
torch.cuda.set_device(cuda_device)

if not os.path.exists("./test_demo/verilog"):
    os.makedirs("./test_demo/verilog")

In [20]:
# Fetch the dataset from OpenML
dataset = openml.datasets.get_dataset(42468)
df_features, df_labels, _, attribute_names = dataset.get_data(dataset_format='dataframe', target=dataset.default_target_attribute)

features = df_features.values.astype(np.float32)

label_names = list(df_labels.unique())
labels = np.array(df_labels.map(lambda x: label_names.index(x)).values)
num_output = labels.max() + 1
input_size = features.shape[1]
output_size = num_output

# Convert data to PyTorch tensors
tensor_features = torch.tensor(features)
tensor_labels = torch.tensor(labels)

data = TensorDataset(tensor_features, tensor_labels)
train_size = int(0.8 * len(data))
val_size = len(data) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(data, [train_size, val_size])

dataset = {}
dataset["train"] = train_dataset
dataset["valid"] = train_dataset
dataset["test"] = val_dataset

test_loader = DataLoader(
        dataset["test"], batch_size=config["batch_size"], shuffle=False
    )

In [21]:
imask = torch.load("./test_demo/imask.pth", map_location="cuda:{}".format(cuda_device))

In [22]:
model_cfg = {}
for k in model_config.keys():
    model_cfg[k] = config[k]
model_cfg["input_length"] = 16
model_cfg["output_length"] = 5
model_cfg["imask"] = torch.load("./test_demo/imask.pth", map_location="cuda:{}".format(cuda_device))
model_cfg['dense_forward'] = False
model_cfg["cuda"] = True

In [23]:
model = JetSubstructureNeqModel(model_cfg)
model.cuda()

Creating input layer
Creating support layer
Creating support layer
Creating support layer
Creating support layer
Creating support layer
Creating output layer


JetSubstructureNeqModel(
  (module_list): ModuleList(
    (0): SparseLinearNeq(
      (input_quant): QuantBrevitasActivation(
        (brevitas_module): QuantHardTanh(
          (input_quant): ActQuantProxyFromInjector(
            (_zero_hw_sentinel): StatelessBuffer()
          )
          (act_quant): ActQuantProxyFromInjector(
            (_zero_hw_sentinel): StatelessBuffer()
            (fused_activation_quant_proxy): FusedActivationQuantProxy(
              (activation_impl): Identity()
              (tensor_quant): RescalingIntQuant(
                (int_quant): IntQuant(
                  (float_to_int_impl): RoundSte()
                  (tensor_clamp_impl): TensorClamp()
                  (delay_wrapper): DelayWrapper(
                    (delay_impl): _NoDelay()
                  )
                )
                (scaling_impl): ParameterScaling(
                  (restrict_clamp_scaling): _RestrictClampValue(
                    (clamp_min_ste): Identity()
               

In [24]:
checkpoint = torch.load("./test_demo/best_accuracy.pth", map_location="cuda:{}".format(cuda_device))
model.load_state_dict(checkpoint["model_dict"])

<All keys matched successfully>

In [25]:
# Test the PyTorch model
print("Running inference on baseline model...")
baseline_accuracy = test(model, test_loader, cuda=True)
print("Baseline accuracy: %f" % (baseline_accuracy))

Running inference on baseline model...


Baseline accuracy: 75.993976


In [26]:
# Generate the truth tables in the LUT module
print("Converting to NEQs to LUTs...")
generate_truth_tables(model, verbose=True)

Converting to NEQs to LUTs...
Calculating truth tables for module_list.0
Truth tables generated for 320 neurons
Calculating truth tables for module_list.1
Truth tables generated for 160 neurons
Calculating truth tables for module_list.2
Truth tables generated for 80 neurons
Calculating truth tables for module_list.3
Truth tables generated for 40 neurons
Calculating truth tables for module_list.4
Truth tables generated for 20 neurons
Calculating truth tables for module_list.5
Truth tables generated for 10 neurons
Calculating truth tables for module_list.6
Truth tables generated for 5 neurons


In [27]:
# Test the LUT-based model
print("Running inference on LUT-based model...")
lut_inference(model)
lut_accuracy = test(model, test_loader, cuda=True)
print("LUT-Based Model accuracy: %f" % (lut_accuracy))
modelSave = {"model_dict": model.state_dict(), "test_accuracy": lut_accuracy}
torch.save(modelSave, "./test_demo/verilog/" + "/lut_based_model.pth")

Running inference on LUT-based model...
LUT-Based Model accuracy: 75.996386


In [28]:
print("Generating verilog in %s..." % ("./test_demo/verilog/"))
module_list_to_verilog_module(
    model.module_list,
    "neuralut",
    "./test_demo/verilog/",
    add_registers=True,
)
print("Top level entity stored at: %s/neuralut.v ..." % ("./test_demo/verilog/"))

Generating verilog in ./test_demo/verilog/...
Top level entity stored at: ./test_demo/verilog//neuralut.v ...


In [29]:
import os

os.environ["OHMYXILINX"] = "/home/ma11418/NeuraLUT_Private/NeuraLUT_Private/oh-my-xilinx"

In [30]:
print("Running out-of-context synthesis")
ret = synthesize_and_get_resource_counts("./test_demo/verilog/", "neuralut", fpga_part='xcvu9p-flgb2104-2-i', clk_period_ns='1.1', post_synthesis=1)
print("Max f: " + str(ret))

Running out-of-context synthesis


/home/ma11418/NeuraLUT_Private/NeuraLUT_Private/oh-my-xilinx/vivadoprojgen.sh:31: no matches found: ../*.vhd
/home/ma11418/NeuraLUT_Private/NeuraLUT_Private/oh-my-xilinx/vivadoprojgen.sh:33: no matches found: ../*.h
/home/ma11418/NeuraLUT_Private/NeuraLUT_Private/oh-my-xilinx/vivadoprojgen.sh:34: no matches found: ../*.xdc
/home/ma11418/NeuraLUT_Private/NeuraLUT_Private/oh-my-xilinx/vivadoprojgen.sh:35: no matches found: ../*.vh
cat: neuralut.xdc: input file is output file


['LUT', '1866']
['FF', '1995']
['DSP', '0']
['BRAM', '0']
['WNS', '0.129']
['']
Max f: {'vivado_proj_folder': './test_demo/verilog//results_neuralut', 'LUT': 1866.0, 'FF': 1995.0, 'DSP': 0.0, 'BRAM': 0.0, 'WNS': 0.129, '': 0, 'fmax_mhz': 1029.8661174047372}
